# Synergy prediction using bulk RNA-seq data

## Data loading

In [4]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [5]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    hsa_score,
    get_all_synergy_data
)

# Local data directories
l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge_timezero"
cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"

# Bliss
bliss_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# HSA
hsa_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = hsa_score,
    time_matched = True
)

## Data cleaning

Ideas:
- Look at which genes have NA values
- Look at which features are most correlated with synergy
- Explore more feature engineering ideas

## Training with stratified split

Drop genes with NA values.

In [3]:
# Try dropping the na columns aggressively
filtered_bliss = bliss_df.dropna(axis = 1)
filtered_hsa = hsa_df.dropna(axis = 1)

Nested CV for ElasticNet.

In [29]:
from src.split import combination_stratified_split
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV

# Define a function to run nested EN CV
def run_nested_elasticnet_cv(df, splits):

    scores = []

    for train_idx, test_idx in splits:

        # Separate train and test data
        train_df = df.iloc[train_idx]
        X_train = train_df.iloc[:, train_df.columns.str.contains("SP")]
        y_train = train_df["synergy_score"]

        test_df = df.iloc[test_idx]
        X_test = test_df.iloc[:, test_df.columns.str.contains("SP")]
        y_test = test_df["synergy_score"]

        # Pipeline for ElasticNet
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNetCV(cv = 5))
        ])

        # Fit and predict
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        # Evaluate
        score = r2_score(y_test, preds)
        scores.append(score)
    
    mean_score = np.mean(scores)

    # Round
    scores = [round(score, 3) for score in scores]
    mean_score = round(mean_score, 3)

    return scores, mean_score
    
# Make stratified splits
strat_splits = combination_stratified_split(filtered_bliss, num_folds = 5, seed = 111)

# Run Nested CV
scores1, mean_score1 = run_nested_elasticnet_cv(
    df = filtered_bliss,
    splits = strat_splits
)

print(f"R^2 for 5 folds : {scores1}")
print(f"Mean R^2 : {mean_score1}")

R^2 for 5 folds : [0.484, 0.428, 0.712, 0.555, 0.515]
Mean R^2 : 0.539


Nested CV for PLS regression.

In [30]:
def run_nested_pls_cv(df, splits):

    scores = []

    for train_idx, test_idx in splits:
        train_df = df.iloc[train_idx]
        X_train = train_df.iloc[:, train_df.columns.str.contains("SP")]
        y_train = train_df["synergy_score"]

        test_df = df.iloc[test_idx]
        X_test = test_df.iloc[:, test_df.columns.str.contains("SP")]
        y_test = test_df["synergy_score"]

        # Create a nested hyperparameter tuning scheme
        param_grid = {
            "model__n_components": list(range(3, 20))
        }

        # Make pipeline for PLS regression
        pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", PLSRegression())
        ])

        # Setup GridSearch
        search = GridSearchCV(
            estimator = pipeline,
            cv = 5,
            param_grid = param_grid,
            scoring = "neg_mean_squared_error",
        )

        # Fit with best params
        search.fit(X_train, y_train)
        preds = search.predict(X_test)

        # Evaluate
        score = r2_score(y_test, preds)
        scores.append(score)
    
    mean_score = np.mean(scores)

    # Round
    scores = [round(score, 3) for score in scores]
    mean_score = round(mean_score, 3)

    return scores, mean_score

scores2, mean_score2 = run_nested_pls_cv(
    df = filtered_bliss,
    splits = strat_splits
)

print(f"R^2 for 5 folds : {scores2}")
print(f"Mean R^2 : {mean_score2}")

R^2 for 5 folds : [0.566, 0.479, 0.822, 0.712, 0.75]
Mean R^2 : 0.666


## Training with held-out drug combination

Nested CV for ElasticNet.

In [31]:
from src.split import combination_held_out_split

# Make held out splits
held_out_splits = combination_held_out_split(filtered_bliss)

scores3, mean_score3 = run_nested_elasticnet_cv(
    df = filtered_bliss,
    splits = held_out_splits
)

print(f"R^2 for 5 folds : {scores3}")
print(f"Mean R^2 : {mean_score3}")

R^2 for 5 folds : [-9.741, -4.332, -1.065]
Mean R^2 : -5.046


Nested CV for PLS regression.

In [32]:
scores4, mean_score4 = run_nested_pls_cv(
    df = filtered_bliss,
    splits = held_out_splits
)

print(f"R^2 for 5 folds : {scores4}")
print(f"Mean R^2 : {mean_score4}")

R^2 for 5 folds : [-18.531, -3.542, -0.816]
Mean R^2 : -7.629


## Training with held-out timepoint

NestedCV for ElasticNet.

In [33]:
from src.split import timepoint_held_out_split

# Make time splits
time_splits = timepoint_held_out_split(filtered_bliss)

scores5, mean_score5 = run_nested_elasticnet_cv(
    df = filtered_bliss,
    splits = time_splits
)

print(f"R^2 for 5 folds : {scores5}")
print(f"Mean R^2 : {mean_score5}")

c:\Users\eddyk\miniconda3\envs\cfu-regression\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.534e-03, tolerance: 1.067e-03
  model = cd_fast.enet_coordinate_descent(


R^2 for 5 folds : [0.05, 0.073, -289.819]
Mean R^2 : -96.566


NestedCV for PLS regression.

In [35]:
time_splits = timepoint_held_out_split(filtered_bliss)

scores6, mean_score6 = run_nested_pls_cv(
    df = filtered_bliss,
    splits = time_splits
)

print(f"R^2 for 5 folds : {scores6}")
print(f"Mean R^2 : {mean_score6}")

R^2 for 5 folds : [-0.03, 0.111, -513.027]
Mean R^2 : -170.982


In [36]:
filtered_bliss["drug_id"].unique()

<ArrowStringArray>
['CEF+CIP', 'CEF+RIF', 'CIP+VNC']
Length: 3, dtype: str